In [1]:
#Copi
import sys
import pandas as pd
import numpy as np
from datetime import datetime
from openpyxl import load_workbook

import os
# Asegúrate de que esta importación sea válida en tu entorno
# from formato_template import exportar_template

In [2]:
# --- Paso 1: Leer Remittance ---
remittance = pd.read_excel(
    "Remittance_copi.xlsx", skiprows=1, nrows=2000,
    usecols=["Referencia", "Clase", "Importe en ML","Texto"]
)

In [3]:
# --- Eliminar guiones de la columna 'Referencia' ---
remittance["Referencia"] = remittance["Referencia"].astype(str).str.replace("-", "", regex=False)
# --- Renombrar Columnas
remittance = remittance.rename(columns={
    "Referencia": "Referencia / Factura",
    "Clase": "Tipo de Documento",
    "Importe en ML": "Importe de factura",
})
remittance = remittance.dropna(subset=["Tipo de Documento"])


# --- Intercambiar signo de los valores
remittance["Importe de factura"] = remittance["Importe de factura"] * -1 # Ver corre en Mac no en Windows
# --- Definición de Reglas (CARDs)
for col in ["Descuento", "Motivo del descuento"]:
    if col not in remittance.columns:
        remittance[col] = ""
conds = [
    #(remittance["Tipo de Documento"].str.startswith("Factura Acrededor", na=False)) & (remittance["Importe de factura"] < 0),
    remittance["Tipo de Documento"].str.startswith("Devolucion", na=False),
    remittance["Tipo de Documento"].str.startswith("Reduc Factura Compra", na=False),
    remittance["Tipo de Documento"].str.startswith("Traslado Notas  Deudor acreedor", na=False),       
]
descuentos = ["AVERIA", "DESCUENTO", "FACT PROVEEDOR"]
motivos = ["522", "987", "CSB"]
remittance["Descuento"] = np.select(conds, descuentos, default=remittance["Descuento"])
remittance["Motivo del descuento"] = np.select(conds, motivos, default=remittance["Motivo del descuento"])
# --- Condición adicional para textos que comienzan con "DCTO 2.00%" ---
mask_dcto = remittance["Texto"].astype(str).str.startswith("DCTO 2.00%")
remittance.loc[mask_dcto, "Descuento"] = "DPP NO PROCEDE"
remittance.loc[mask_dcto, "Motivo del descuento"] = "667"
# --- Condición adicional para textos que comienzan con "Dev.>" ---
mask_dev = remittance["Texto"].astype(str).str.startswith("Dev.>")
remittance.loc[mask_dev, "Descuento"] = "AVERIA"
remittance.loc[mask_dev, "Motivo del descuento"] = "522"
# --- Condición para las Notas que no estan con datos en descuento y motivo de descuento
condicion = (remittance["Tipo de Documento"] == "Nota") & (remittance["Descuento"].astype(str).str.strip() == "")
remittance.loc[condicion, "Descuento"] = "DESCUENTO"
remittance.loc[condicion, "Motivo del descuento"] = 987

In [4]:
remittance["Tipo de Documento"] = remittance["Tipo de Documento"].replace({
    "Factura Acrededor": "Factura",
    "Reduc Factura Compra": "Descuentos no asociados a FC",
    "Nota": "Descuentos no asociados a FC",
    "Devolucion": "Descuentos no asociados a FC"
})

In [5]:
remittance.head()

,Referencia / Factura,Tipo de Documento,Importe de factura,Texto,Descuento,Motivo del descuento
0,PMP1276670,Descuentos no asociados a FC,-4956.0,NaN,DESCUENTO,987
1,PMP1276670,Factura,8387269.0,NaN,,
2,PMP1276992,Descuentos no asociados a FC,-4956.0,UNILEVER ANDINA COLOMBIA LTDA/conciliaicon,DESCUENTO,987
3,PMP1277139,Factura,20008459.0,UNILEVER ANDINA COLOMBIA LTDA,,
4,PMP1276992,Factura,19850219.0,UNILEVER ANDINA COLOMBIA LTDA/conciliaicon,,


In [ ]:
remittance_factura = remittance[remittance["Tipo de Documento"] == "Factura"]
remittance_desc = remittance[remittance["Tipo de Documento"] != "Factura"]

In [13]:
# --- Paso 5: Leer FBL5N ---
FBL5N = pd.read_excel(
    "FBL5N_copi.xlsx",
    sheet_name="Sheet1",
    usecols=["Document Type", "Reference", "Amount in local currency", "Reason code", "Document Number", "Text"]
)
FBL5N = FBL5N[(FBL5N["Document Type"] == "RV") | (FBL5N["Reason code"] == "NRO")]
FBL5N = FBL5N.rename(columns={
    "Reference": "Referencia / Factura",
    "Amount in local currency": "importe_FBL5N"
}).reset_index(drop=True)
# --- Incluir partidas NRO de la FBL5N
FBL5N["Referencia / Factura"] = np.where(
    FBL5N["Reason code"] == "NRO",
    FBL5N["Document Number"].astype("Int64").astype(str),
    FBL5N["Referencia / Factura"]
)

In [ ]:
    # ---  Merge ---
hrc_template = pd.merge(remittance_factura, FBL5N, on="Referencia / Factura", how="left")

In [ ]:
hrc_template = pd.concat([hrc_template, remittance_desc], ignore_index=True)

In [18]:

    
    hrc_template["Referencia / Factura"] = hrc_template["Referencia / Factura"].str.replace(r"^NC-", "", regex=True)

    # ---  Diferencias ---
    hrc_template["Diferencia"] = pd.NA
    hrc_template.loc[hrc_template["Tipo de Documento"] == "Factura", "Diferencia"] = (
        hrc_template["importe_FBL5N"] - hrc_template["Importe de factura"]
    )

    diferencias = hrc_template[hrc_template["Diferencia"].notna() & (hrc_template["Diferencia"] != 0)].copy()
    registros_diferencias = pd.DataFrame({
        "Referencia / Factura": diferencias["Referencia / Factura"],
        "Texto": diferencias["Texto"],
        "Importe de factura": diferencias["Diferencia"],
        "Tipo de Documento": "Descuentos no asociados a FC",
        "Pago Neto": "",
        "Descuento": "MENORES VALORES",
        "Motivo del descuento": np.select(
            condlist=[
                (diferencias["Diferencia"] <= -20000) | (diferencias["Diferencia"] >= 20000),
                (diferencias["Diferencia"].between(-20000, 0, inclusive="neither")),
                (diferencias["Diferencia"].between(0, 20000, inclusive="left"))
            ],
            choicelist=["987", "WOB", "384"],
            default="Error (Revisar)"
        )
    })

    hrc_template = pd.concat([hrc_template, registros_diferencias], ignore_index=True)

In [19]:
registros_diferencias.head(100)

,Referencia / Factura,Texto,Importe de factura,Tipo de Documento,Pago Neto,Descuento,Motivo del descuento
